# 08 · Conversation Memory Patterns

The checkpointer (lesson 07) stores the **full transcript** — and re-sends it every call. Cost and latency grow every turn, and quality drops on long stale context.

The fix: separate **what you store** (everything) from **what the model sees** (a curated view):

```
window     →  keep the last N messages
token trim →  keep what fits a token budget
summary    →  compress old turns, keep recent ones verbatim
```

---

## Setup

In [ ]:
%pip install -qU langchain langchain-openai python-dotenv

In [ ]:
import os
from langchain.chat_models import init_chat_model
from langchain_core.messages import (
    SystemMessage, HumanMessage, AIMessage, trim_messages,
)
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from dotenv import load_dotenv

load_dotenv("../../.env")

llm = init_chat_model(os.environ["CHAT_MODEL"], temperature=0)

def approx_tokens(messages) -> int:
    """Rough token estimate (~4 chars/token) — swap for a real tokenizer in production."""
    return sum(len(m.content) // 4 for m in messages)

---
## 1. The problem, measured

A conversation that just appends: every turn re-sends everything before it.

In [ ]:
history = [SystemMessage("You are a helpful travel assistant.")]

for turn in [
    "Hi! I'm Harish, planning a 10-day trip to Japan in November.",
    "I prefer trains over flights, and my budget is around $2500.",
    "I want to see Tokyo, Kyoto and one quiet mountain town.",
    "I'm vegetarian, so food recommendations matter a lot.",
]:
    history.append(HumanMessage(turn))
    reply = llm.invoke(history)
    history.append(AIMessage(reply.content))
    print(f"turn {len(history)//2}: history = {len(history):>2} messages, "
          f"~{approx_tokens(history):>5} tokens sent")

> Four turns in and the input size has multiplied. At turn 50 you're paying for turns 1–49 *again* — until the context window ends the conversation for you.

---
## 2. Windowing: keep the last N messages

`trim_messages` with `token_counter=len` counts *messages*. Note the two safety flags: never drop the system prompt, and cut on a valid `human` boundary.

In [ ]:
window = trim_messages(
    history,
    strategy="last",       # keep the most recent...
    token_counter=len,     # ...counting messages (1 each)
    max_tokens=4,          # keep 4 messages
    include_system=True,   # the system prompt always survives
    start_on="human",      # views must start on a human turn
)

for m in window:
    print(f"{m.type:>6}: {m.content[:70]}")

---
## 3. Token trimming: respect a real budget

Messages vary in size, so budget in tokens. `token_counter` accepts any `fn(messages) -> int` — or the `llm` itself.

In [ ]:
budgeted = trim_messages(
    history,
    strategy="last",
    token_counter=approx_tokens,
    max_tokens=300,               # a real token budget
    include_system=True,
    start_on="human",
)

print(f"full history: {len(history)} msgs (~{approx_tokens(history)} tokens)")
print(f"within budget: {len(budgeted)} msgs (~{approx_tokens(budgeted)} tokens)")

**The weakness of both:** whatever falls outside the window is *gone*. Ask about the budget from turn 2 with only a 4-message window and the model has never heard of it.

In [ ]:
question = HumanMessage("What was my total budget again?")

view = trim_messages(history + [question], strategy="last", token_counter=len,
                     max_tokens=4, include_system=True, start_on="human")
print(llm.invoke(view).content)

---
## 4. Summarization: compress, don't discard

Fold the older turns into one summary message; keep the recent turns verbatim. Facts survive — compressed.

In [ ]:
summarizer = (
    ChatPromptTemplate.from_template(
        "Summarize this conversation in 3-4 sentences, keeping every concrete fact "
        "(names, numbers, dates, preferences):\n\n{conversation}"
    )
    | llm
    | StrOutputParser()
)

def summarize_view(history, keep_last=2):
    """Compress everything but the last `keep_last` messages into a summary."""
    system, rest = history[0], history[1:]
    old, recent = rest[:-keep_last], rest[-keep_last:]
    if not old:
        return history
    transcript = "\n".join(f"{m.type}: {m.content}" for m in old)
    summary = summarizer.invoke({"conversation": transcript})
    return [system, SystemMessage(f"Summary of the conversation so far:\n{summary}")] + list(recent)

view = summarize_view(history)
print(f"full: {len(history)} msgs (~{approx_tokens(history)} tokens)  →  "
      f"view: {len(view)} msgs (~{approx_tokens(view)} tokens)\n")
print(view[1].content)

Same question that windowing failed — the fact from turn 2 now survives inside the summary:

In [ ]:
print(llm.invoke(view + [question]).content)

---
## 5. The hybrid most assistants run

Summary of the old + exact recent window, rebuilt per call. Store everything; show what matters.

In [ ]:
def chat(history, user_input, keep_last=4):
    """One turn: store the full history, but send the model a compact view."""
    history.append(HumanMessage(user_input))
    view = summarize_view(history, keep_last=keep_last)
    reply = llm.invoke(view)
    history.append(AIMessage(reply.content))
    print(f"[sent {len(view)} msgs / ~{approx_tokens(view)} tokens "
          f"(full: {len(history)} msgs / ~{approx_tokens(history)})]")
    return reply.content

print(chat(history, "Given everything so far, suggest the quiet mountain town."))

> Inside a `create_agent`, this exact logic mounts as **middleware** (e.g. `SummarizationMiddleware`) so it runs before the model automatically — same pattern, different mounting point.

---
## What to remember

| Concept | What it does |
|---|---|
| Store vs. see | Keep the full transcript; curate the model's view per call |
| `trim_messages(token_counter=len)` | Window: last N messages |
| `trim_messages(token_counter=fn/llm)` | Token budget: keep what fits |
| `include_system=True`, `start_on="human"` | Don't lose instructions; cut on valid boundaries |
| Summarize old + keep recent | Facts survive compression; cost stays flat |
| Hybrid (summary + window) | The default shape for long-running assistants |

**Next:** 09 — **RAG, naive to advanced**: the same idea for *documents* — retrieve what matters instead of stuffing everything in.